## **Reading Food Hygiene Data Analysis**

### Objectives

* Analyse the UK Food Standards Agency dataset, and clean in preparation for data analysis.

### Planned Steps
1. Download and load raw data
2. Filter data to only Reading Local Authority
3. Identify columns needed for analysis - Create new Reading dataset
4. Initial inspection of data
5. Conclusion of data inspection and cleaning strategy
6. Clean data and validate
7. Export cleaned dataset for analysis, visualisation and conclusion

### Inputs (Data Source)
* **Data Source:** [UK Food Standards Agency - Open Data CSV](https://safhrsprodstorage.blob.core.windows.net/opendatafileblobstorage/FHRS_All_en-GB.csv)

### Outputs
* (3) - Filtered Reading Local Authority dataset
* (6) - Cleaned Reading Local Authority dataset for analysis

---

### Setup
Install and Import required libraries

In [ ]:
%pip install pandas numpy

import numpy as np
import pandas as pd

print("Libraries imported successfully!")

---
### 1. Download and load raw data.
Raw data downloaded from the Food Standards Agency (link above).

**File Location:** Saved locally in the `data` directory as `FHRS_All_en-GB.csv`

In [ ]:
file_path = "data/FHRS_All_en-GB.csv"  # Set the file path to the CSV file
print(f"Reading data from {file_path}...")  # Print a message indicating the file being read
df_raw = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)  # Read the CSV file into a DataFrame
print(f"Data read successfully. Shape of the DataFrame: {df_raw.shape}")  # Print the shape of the DataFrame

---
### 2. Filter data to only Reading Local Authority
- Filter data to Reading
- Look through both outputs to identify any duplicates or things that could become blockers in moving forward.

In [ ]:
# Identify the name of the columns to target local authority
df_raw.columns.tolist()  

In [ ]:
# Identify the actual name of the local authority, and sort alphabetically

unique_authorities = df_raw['LocalAuthorityName'].unique().tolist()  # List of unique local authorities
unique_authorities.sort()  # Sort the list alphabetically 
unique_authorities

---
 ### 3. Identify columns needed for analysis
We already know the names of the columns from Step 2.

The columns to keep:
| Column Name | What it is | Rationale |
| :--- | :--- | :--- |
| LocalAuthorityBusinessID | Unique reference code given to the establishment | Serves as a unique identifier for the business
| BusinessName | Trading name of the business | Essential for labelling charts and specific lookups
| BusinessType | The category of the business (e.g., Restaurant/Cafe/Retailer) | Crucial for comparative analysis across sectors
| RatingValue | The hygiene score awarded to the business | Primary target metric for distributions and profiling
| RatingDate | The date when the inspection took place | Enables analysis of inspection trends
| AddressLine1 | The first line of the address for the business | Adds granular location context for individual venues 
| Postcode | The postcode for the business | Useful for neighbourhood grouping and mapping
| LocalAuthorityName | The name of the Local Authority (Reading) | Verification anchor confirming filter accuracy

Save new dataset

In [ ]:
reading_data = (df_raw[df_raw['LocalAuthorityName'] == 'Reading']  # Filter the DataFrame for Reading local authority
[['LocalAuthorityBusinessID', 'BusinessName', 'BusinessType', 'RatingValue', 'RatingDate', 'AddressLine1', 'PostCode', 'LocalAuthorityName']].copy())  # Select relevant columns

reading_data.to_csv('data/reading_food_hygiene_data.csv', index=False)  # Save the filtered DataFrame to a new CSV file

display(reading_data.head())  # Display the first few rows of the filtered DataFrame

---
### Dataset Optimisation
By filtering the nationwide UK Food Standards Agency raw dataset **(137MB)**, to our target area of **Reading** and selecting only the 8 target columns, the processed dataset has been streamlined to just **143KB**.

**Why this matters:** This drastically speeds up execution times for visualisation tools, while keeping our project lightweight. 

**File Location:** Saved locally in the `data` directory as `reading_food_hygiene_data.csv`

As this file is significantly smaller, it then gave me a chance to open the file in Microsoft Excel, to inspect the file for missing data and any significant problems before moving any further.

---
### 4. Initial inspection of data
- Load `reading_food_hygiene_data.csv`
- Check the initial shape of the dataset
- Check the columns, data type and non-null counts
- Check for missing values across all columns
- Check the percentage of missing data for each column
- Check the summary statistics for each column
- Consider missing data and how to resolve it

In [ ]:
file_path = "data/reading_food_hygiene_data.csv"  # Set the file path to the new CSV file
df = pd.read_csv(file_path)  # Read the CSV file into a DataFrame

print(f'Shape:\n{df.shape}')  # Print the initial shape of the dataframe

In [ ]:
print(df.dtypes)  # Print columns and data types of each

In [ ]:
df.isnull().sum()  # Print the number of null values in each column

In [ ]:
df.isnull().sum() / len(df) * 100  # Print the percentage of null values in each column

In [ ]:
df.describe(include='all') # Check summary statistics for all columns

---
### Missing Data
All columns except three contain data

By the above, we can see that we are missing values in the following columns:
- **188** in RatingDate - These equate to 12.46%
- **158** in AddressLine1 - These equate to 10.47%
- **187** in PostCode - These equate to 12.40%

We now need to look at these columns to check what the missing data is, and make a decision on what to do with them

In [ ]:
# Filter to the rows where any columns have null values
missing_rows = df[df['RatingDate'].isnull() | df['AddressLine1'].isnull() | df['PostCode'].isnull()]
missing_rows


**Initial Thoughts - Missing RatingDate**
- The businesses that are missing ***RatingDate*** values appear to be due to ***AwaitingInspection*** or ***Exempt***.
  - Check to see if the value of the missing *RatingDate* correlates to *AwaitingInspection* or *Exempt*.
  - Re-check if RatingDate is then zero afterwards.
  - **Consider removal** - as even though they equate for 12.46%, they serve no value to answering our questions or helping to visualise the dataset.

In [ ]:
# Check which businesses have a missing RatingDate, against the rating values (AwaitingInspection and Exempt).
unrated_or_exempt = df[(df['RatingDate'].isnull()) & (df['RatingValue'].isin(['AwaitingInspection', 'Exempt']))]
unrated_or_exempt

In [ ]:
# Count the remaining null values in the DataFrame
remaining_nulls = df.drop(unrated_or_exempt.index).isnull().sum()
remaining_nulls

**Initial Thoughts - AddressLine1 and PostCode**
- Is there a connection between the type of Business, and addresses and postcodes which would explain why this data is missing?
  - Check to see if the value of the missing *AddressLine1* and *PostCode* correlate to a specific *BusinessType*
  - Is there a specific reason? (Home-based/Mobile food businesses - GDPR perhaps?)
  - **Consider imputing missing values with *'Unknown'*** - This would work for both *AddressLine1* and *PostCode*, as regardless of the missing data, the remaining that is populated would help support answering our **Key Analytical Questions**, and visualisations.
  - Consider adding an additional Postcode District (e.g., RG1, RG2), to enable regional visualisations.

In [ ]:
# Count of missing address and postcodes grouped by business type
missing_address = df[df['AddressLine1'].isnull() | df['PostCode'].isnull()]['BusinessType'].value_counts()
missing_address

In [ ]:
# Look at breakdown of missing addresses, to consider business names
pd.set_option('display.max_rows', None)
missing_businesses = df[df['AddressLine1'].isnull() | df['PostCode'].isnull()][['BusinessName', 'BusinessType', 'AddressLine1', 'PostCode']]
missing_businesses

---
### 5. Conclusion of data inspection and cleaning strategy

Based on the initial inspection of the dataset, the following observations and data gaps were identified, leading to this cleaning strategy.

* **Missing / Unrated Data:** Rows missing a rating or inspection date will be dropped, as they lack the scoring information required for analysis.

* **Missing Addresses and Postcodes:** Null values in `AddressLine1` and `Postcode` will be imputed with `Unknown` to preserve the valid hygiene score records.

* **Postcode Feature Engineering:** Full postcodes will be split to extract the district within Reading (e.g., `RG1`, `RG2`) into a new `PostCodeDistrict` column to enable cleaner regional visualisations.

* **Data Type Conversions:** Hygiene ratings will be converted from strings to integers to allow for proper numerical calculations and visualisations.

* **Date formatting:** The `RatingDate` column will be changed to a standard date format to support tracking trends over time.

---
### 6. Clean data and validate

Following the cleaning strategy above, we will now apply the cleaning steps.

In [ ]:
# Create a working copy for cleaning
df_rfh_clean = df.copy()

**Missing / Unrated Data:** Rows missing a rating or inspection date will be dropped, as they lack the scoring information required for analysis.

In [ ]:
# Check shape before cleaning
print(df_rfh_clean.shape)

# Drop rows missing a rating or inspection date
df_rfh_clean = df_rfh_clean.dropna(subset=['RatingValue', 'RatingDate'])

# Validation - Check removal of rows
print(df_rfh_clean.shape)

**Missing Addresses and Postcodes:** Null values in `AddressLine1` and `Postcode` will be imputed with `Unknown` to preserve the valid hygiene score records.

In [ ]:
# Impute missing values in AdressLine1 and Postcode with 'Unknown'
df_rfh_clean['AddressLine1'] = df_rfh_clean['AddressLine1'].fillna('Unknown')
df_rfh_clean['PostCode'] = df_rfh_clean['PostCode'].fillna('Unknown')

# Validation - Check remaining null values
print(df_rfh_clean.isnull().sum())  # Print the number of null values in each column

**Postcode Feature Engineering:** Full postcodes will be split to extract the district within Reading (e.g., `RG1`, `RG2`) into a new `PostCodeDistrict` column to enable cleaner regional visualisations.

In [ ]:
# Split postcodes to give a new column of PostCodeDistrict
df_rfh_clean['PostCodeDistrict'] = df_rfh_clean['PostCode'].apply(lambda x: str(x).split(' ')[0] if pd.notnull(x) else 'Unknown')

# Validate the addition
df_rfh_clean.head()

**Data Type Conversions:** Hygiene ratings will be converted from strings to integers to allow for proper numerical calculations and visualisations.

In [ ]:
df_rfh_clean['RatingValue'] = df_rfh_clean['RatingValue'].astype(int)

# Validate the data type conversion
df_rfh_clean['RatingValue'].dtype

**Date formatting:** The `RatingDate` column will be changed to a standard date format to support tracking trends over time.

In [ ]:
# Convert RatingDate to datetime format
df_rfh_clean['RatingDate'] = pd.to_datetime(df_rfh_clean['RatingDate'])

# Validate the conversion
df_rfh_clean['RatingDate'].dtype

---
### Validation
Now the data cleaning steps have been completed. I now need to carry out a final validation check.

This ensures that the cleaned dataset is correctly structured, and reliable before moving forward to data analysis and visualisations.

In [425]:
# Check the rows and number of columns
df_rfh_clean.shape

(1320, 9)

In [426]:
# Check the data types of each column
df_rfh_clean.dtypes

LocalAuthorityBusinessID               str
BusinessName                           str
BusinessType                           str
RatingValue                          int64
RatingDate                  datetime64[us]
AddressLine1                           str
PostCode                               str
LocalAuthorityName                     str
PostCodeDistrict                       str
dtype: object

In [427]:
# Check the number of null counts in each column
df_rfh_clean.isnull().sum()

LocalAuthorityBusinessID    0
BusinessName                0
BusinessType                0
RatingValue                 0
RatingDate                  0
AddressLine1                0
PostCode                    0
LocalAuthorityName          0
PostCodeDistrict            0
dtype: int64

The validation checks confirm that the cleaning strategy has been carried out successfully on `df_rfh_clean`. 

The missing values have been handled, data types (including datetime formatting) are correctly structured, and the new postcode district column has been created.